# The code represents RAG with hybrid search (dense + sparse) pipeline from data ingestion to llm answer generation

**1) Install libraries**

In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain_groq chromadb sentence-transformers transformers groq langchain-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.2 MB/s eta

**2) Import all necessary packages**

In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage
from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
import heapq
from collections import defaultdict
from getpass import getpass

**3) Load the document**

In [ ]:
# Load Document
Loader = TextLoader('/content/Healthy_Eating_for_Heart_Disease_Prevention.txt')
docs = Loader.load()

**4) Chunk the documents using chunk overlap**

In [ ]:
# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)
chunks = text_splitter.split_documents(docs)

In [ ]:
# Length of the chunks
print(len(chunks))

10


In [ ]:
# Display some chunks
print(chunks[0])
print(chunks[1])

page_content='Healthy Eating for Heart Disease Prevention

A well-balanced diet is one of the most effective tools for preventing heart disease and promoting overall cardiovascular health. The right dietary choices can help manage blood pressure, cholesterol, blood sugar levels, and weight—all key factors in reducing the risk of heart disease. Below is a more detailed guide to help you make heart-healthy food choices.

1. Focus on Heart-Healthy Fats
Fat is an essential nutrient that the body needs for energy, cell growth, and absorption of fat-soluble vitamins. However, not all fats are the same. Healthy fats can help protect your heart, while unhealthy fats increase the risk of heart disease.' metadata={'source': '/content/Healthy_Eating_for_Heart_Disease_Prevention.txt'}
page_content='- Monounsaturated Fats:  
  These fats help lower LDL (bad) cholesterol levels and raise HDL (good) cholesterol levels.  
  - Sources: Olive oil, canola oil, peanut butter, avocados, almonds, and walnut

**5) Dense Retrieval : Vector-based semantic search**

In [ ]:
# Initialize Dense Retriever (Chroma + Sentence Transformers)
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Number of documents in vectorstore: {vectorstore._collection.count()}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of documents in vectorstore: 10


In [ ]:
# Initialize Dense Retriever
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

**6) Sparse Retrieval : keyword based matching**

In [ ]:
# Initialize Sparse Retriever (TF-IDF)
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform([chunk.page_content for chunk in chunks])

In [ ]:
# Function to retrieve top documents using TF-IDF
def sparse_retriever(query, k=4):
    tfidf_query_vector = tfidf_vectorizer.transform([query])
    tfidf_similarities = cosine_similarity(tfidf_query_vector, tfidf_matrix)
    tfidf_top_indices = tfidf_similarities.argsort()[0][-k:][::-1]  # Top k results
    return [chunks[i] for i in tfidf_top_indices]

**7) Reciprocal Rank Fusion**

In [ ]:
# Reciprocal Rank Fusion (RRF) to Combine Dense and Sparse Retrieval
def reciprocal_rank_fusion(dense_docs, sparse_docs, k=60):
    # Combine and rank both dense and sparse results using Reciprocal Rank Fusion
    fused_scores = defaultdict(float)

    # Process dense results
    for i, doc in enumerate(dense_docs):
        fused_scores[doc.page_content] += 1 / (k + i)

    # Process sparse results
    for i, doc in enumerate(sparse_docs):
        fused_scores[doc.page_content] += 1 / (k + i)

    # Rank the documents based on fused scores
    ranked_docs = heapq.nlargest(len(fused_scores), fused_scores.items(), key=lambda x: x[1])
    return [doc[0] for doc in ranked_docs]

In [ ]:
# Format the Retrieved Documents for Display
def format_docs(docs):
    return "\n".join(doc['text'] for doc in docs)

**8) Query Processing**

In [ ]:
query = "I want to eat better to prevent heart disease. What should I avoid?"

# Get Dense Retrieval Results
dense_results = dense_retriever.get_relevant_documents(query)

# Mark Dense Documents
dense_docs = [(doc, 'dense') for doc in dense_results]

# Get Sparse Retrieval Results (TF-IDF)
sparse_results = sparse_retriever(query)

# Mark Sparse Documents
sparse_docs = [(doc, 'sparse') for doc in sparse_results]

# Combine results using Reciprocal Rank Fusion
combined_results = reciprocal_rank_fusion(dense_results, sparse_results)

**9) Display Retrieved Dense and Sparse Docs**

In [ ]:
# Print out Dense and Sparse Docs separately
print("Dense Documents Retrieved:\n")
for i, doc in enumerate(dense_docs):
    print(f"Doc chunk {i+1} (Dense):\n{doc[0].page_content}\n{'-'*40}")

print("\nSparse Documents Retrieved:\n")
for i, doc in enumerate(sparse_docs):
    print(f"Doc chunk {i+1} (Sparse):\n{doc[0].page_content}\n{'-'*40}")

Dense Documents Retrieved:

Doc chunk 1 (Dense):
Sample Heart-Healthy Meal Plan
Breakfast: Oatmeal topped with fresh berries, flaxseeds, and a handful of almonds.  
Lunch: Grilled salmon with a side of mixed greens, quinoa, and olive oil vinaigrette.  
Snack: A banana with a handful of walnuts.  
Dinner: Grilled chicken breast with roasted sweet potatoes, steamed broccoli, and a small serving of brown rice.  
Dessert: A small bowl of mixed berries topped with a dollop of Greek yogurt.

Conclusion
Heart disease is preventable with the right dietary choices. A diet rich in fruits, vegetables, whole grains, healthy fats, and lean proteins can reduce the risk of heart disease and improve overall cardiovascular health. By following these guidelines and making heart-healthy choices, you can take proactive steps toward a healthier heart and a longer, more active life.
----------------------------------------
Doc chunk 2 (Dense):
Healthy Eating for Heart Disease Prevention

A well-balanced die

**10) Augmentation and Generation**

In [ ]:
# Provide context and query as a prompt to the llm
context = "\n".join([doc for doc in combined_results])
full_prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

In [ ]:
# LLM configuration
api_key = getpass("Enter your Groq API key: ")
os.environ["GROQ_API_KEY"] = api_key

chat_model = ChatGroq(
    model_name="openai/gpt-oss-20b"
)

response = chat_model.predict(full_prompt)

Enter your Groq API key: ··········


In [ ]:
# Display the output
print("Generated Answer:", response)

Generated Answer: ### Foods and habits to steer clear of if you want to keep your heart healthy

| What to avoid | Why it matters for heart health | Quick tips to skip it |
|---------------|--------------------------------|-----------------------|
| **Highly processed snacks & desserts** – cakes, cookies, candy, chips, ready‑made pastries | Packed with trans fats, refined carbs, and hidden sugars that raise triglycerides, LDL (“bad”) cholesterol, and blood pressure. | Swap for a handful of nuts, fresh fruit, or homemade whole‑grain crackers. |
| **Sugary beverages** – sodas, energy drinks, sweetened coffee/tea, fruit‑sweetened juices | Add 200–300 calories a day with no nutrition; they spike blood sugar and triglycerides. | Use sparkling water with a splash of fresh lime, or unsweetened herbal tea. |
| **Excessive added salt** – canned soups, processed meats, fast‑food, salted snacks | Raises blood pressure, which forces the heart to work harder and can damage arteries. | Read labels, 